In [1]:
"""
╔══════════════════════════════════════════════════════════════════╗
║     AMAZON SALES ANALYTICS — E-Commerce Intelligence Report      ║
║         Dataset: 2019-2024  |  5,000 Orders  |  15 Columns      ║
╚══════════════════════════════════════════════════════════════════╝
"""
import warnings, os
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import pearsonr
import statsmodels.api as sm
from statsmodels.stats.weightstats import ztest
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_columns", None)
print("✅ Libraries loaded")

✅ Libraries loaded


In [2]:
# -- COLOUR PALETTE -------------------------------------------------------------
from pathlib import Path

PALETTE   = {"primary":"#0D3B66","accent1":"#F4D35E","accent2":"#EE964B",
             "accent3":"#F95738","accent4":"#1B998B","bg":"#0F1924","card":"#16253A",
             "text":"#E8EDF2","grid":"#1E3050"}
SEQ_COLORS = ["#0D3B66","#1B6CA8","#1B998B","#F4D35E","#EE964B","#F95738"]
CAT_COLORS = ["#1B998B","#F4D35E","#EE964B","#F95738","#E9C46A","#2A9D8F",
              "#264653","#E76F51","#A8DADC","#457B9D"]
DIVERG     = [[0,"#0D3B66"],[0.5,"#FAF0CA"],[1,"#F95738"]]

BASE_LAYOUT = dict(
    paper_bgcolor=PALETTE["bg"], plot_bgcolor=PALETTE["card"],
    font=dict(color=PALETTE["text"], family="Inter, Arial, sans-serif", size=13),
    title_font=dict(size=17, color=PALETTE["accent1"]),
    margin=dict(l=60,r=40,t=65,b=55),
    xaxis=dict(gridcolor=PALETTE["grid"], linecolor=PALETTE["grid"]),
    yaxis=dict(gridcolor=PALETTE["grid"], linecolor=PALETTE["grid"]),
    legend=dict(bgcolor="rgba(0,0,0,.3)",bordercolor=PALETTE["grid"],borderwidth=1),
)

def L(fig, title="", **kw):
    fig.update_layout(**BASE_LAYOUT, title=title, **kw)
    return fig

# -- DATA LOADING ---------------------------------------------------------------
notebook_dir = Path.cwd()
data_file = notebook_dir.parent / "data" / "raw" / "amazon_sales.xlsx"
if not data_file.exists():
    data_file = Path("../data/raw/amazon_sales.xlsx")

df = pd.read_excel(data_file)
df.columns = df.columns.str.strip()
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Year"]       = df["Order Date"].dt.year
df["Month"]      = df["Order Date"].dt.month
df["Month_Name"] = df["Order Date"].dt.strftime("%b")
df["Quarter"]    = df["Order Date"].dt.to_period("Q").astype(str)
df["YearMonth"]  = df["Order Date"].dt.to_period("M").astype(str)

for c in ["Quantity Sold","Unit Price","Discount (%)","Total Sales","Profit Margin"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["Revenue"]       = df["Total Sales"]
df["Profit_Amount"] = df["Total Sales"] * df["Profit Margin"] / 100
df.dropna(subset=["Order Date","Total Sales"], inplace=True)

print(f"Shape: {df.shape}")
df.head()

Shape: (5000, 22)


,Order ID,Order Date,Customer ID,Customer Name,Region,Product Category,Product Name,Quantity Sold,Unit Price,Discount (%),Salesperson,Payment Method,Order Status,Total Sales,Profit Margin,Year,Month,Month_Name,Quarter,YearMonth,Revenue,Profit_Amount
0,576ca8db-ad1a-4aa8-9c3a-89102555f200,2019-01-01,48aa940a-5cb5-4103-8d89-19c10fd26aaf,Stephanie Garcia,South America,Home & Kitchen,Speak,9,484.46,21.33,Johnny Marshall,Debit Card,Returned,3430.12,1193.53,2019,1,Jan,2019Q1,2019-01,3430.12,40939.51
1,b7154d7e-f369-4a81-80b1-d36b23cc0420,2019-01-01,8de3545c-78e6-4962-ace5-02e37fe1ce91,Jacob Morales,South America,Books,If,10,179.28,6.43,Bradley Howe,Gift Card,Pending,1677.52,583.70,2019,1,Jan,2019Q1,2019-01,1677.52,9791.68
2,b2852522-fbaa-43fa-b165-bd10e0035591,2019-01-02,6ac41f61-acf6-4e46-8b70-ad00ce01c9d0,John Carroll,North America,Sports,Big,6,185.34,1.57,Johnny Marshall,Amazon Pay,Completed,1094.58,380.86,2019,1,Jan,2019Q1,2019-01,1094.58,4168.82
3,f14eae78-235f-4283-a2cc-6433fd861c14,2019-01-04,04bb6a6a-a62a-4e79-a038-68758f215ccc,Allison Ramirez,Asia,Home & Kitchen,Serious,5,40.98,4.21,Roger Brown,PayPal,Completed,196.27,68.29,2019,1,Jan,2019Q1,2019-01,196.27,134.03
4,c3fcb6f0-a399-47eb-8903-2a354425ea75,2019-01-04,2267ce03-37ba-446a-b279-17ecee550c6e,James Thompson,North America,Toys,Operation,10,459.39,20.50,Kristen Ramos,PayPal,Cancelled,3652.15,1270.78,2019,1,Jan,2019Q1,2019-01,3652.15,46410.79


In [3]:
# ─── EDA OVERVIEW ─────────────────────────────────────────────────────────────
print(f"Shape         : {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"Date range    : {df['Order Date'].min().date()} → {df['Order Date'].max().date()}")
print(f"Total Revenue : ${df['Revenue'].sum():,.0f}")
print(f"Avg Order Val : ${df['Revenue'].mean():,.2f}")
print(f"Unique Cust.  : {df['Customer ID'].nunique():,}")
print(f"Unique Prods  : {df['Product Name'].nunique():,}")
print("\nMissing values:")
miss = df.isnull().sum()
print(miss[miss>0] if miss.sum()>0 else "  None ✅")
print()
df[["Quantity Sold","Unit Price","Discount (%)","Total Sales","Profit Margin"]].describe().round(2)

Shape         : 5,000 rows × 22 cols
Date range    : 2019-01-01 → 2024-12-31
Total Revenue : $5,935,727
Avg Order Val : $1,187.15
Unique Cust.  : 5,000
Unique Prods  : 964

Missing values:
Product Name    1
dtype: int64



,Quantity Sold,Unit Price,Discount (%),Total Sales,Profit Margin
count,5000.00,5000.00,5000.00,5000.00,5000.00
mean,5.52,255.16,14.90,1187.15,413.07
std,2.90,143.17,8.71,988.46,343.94
min,1.00,5.16,0.00,4.61,1.60
25%,3.00,130.11,7.24,370.13,128.79
50%,6.00,257.08,14.87,907.96,315.93
75%,8.00,379.71,22.44,1762.25,613.18
max,10.00,499.91,29.99,4911.92,1709.12


In [4]:
# ═══ PAGE 2: DATA DISTRIBUTION ════════════════════════════════════════════════
# 1. Revenue Histogram + KDE
fig = ff.create_distplot([df["Revenue"].dropna().tolist()],
                          group_labels=["Revenue"],
                          bin_size=500, colors=[PALETTE["accent1"]], show_rug=False)
L(fig, "Revenue Distribution (Histogram + KDE)")
fig.show()

# 2. Box Plots — 5 KPIs
num_cols = ["Total Sales","Unit Price","Quantity Sold","Discount (%)","Profit Margin"]
fig2 = go.Figure()
for i, col in enumerate(num_cols):
    fig2.add_trace(go.Box(y=df[col].dropna(), name=col,
                          marker_color=SEQ_COLORS[i % len(SEQ_COLORS)], boxmean="sd"))
L(fig2, "Numeric KPIs — Box Plots (Outlier Detection)")
fig2.show()

In [5]:
# ═══ PAGE 3: DATA COMPOSITION ══════════════════════════════════════════════════
# Donut — Revenue by Category
cat_rev = df.groupby("Product Category")["Revenue"].sum().reset_index().sort_values("Revenue", ascending=False)
fig = go.Figure(go.Pie(labels=cat_rev["Product Category"], values=cat_rev["Revenue"],
                        hole=0.52, marker=dict(colors=CAT_COLORS[:len(cat_rev)],
                        line=dict(color=PALETTE["bg"], width=2))))
L(fig, "Revenue Composition by Product Category")
fig.show()

# Treemap — Region → Category
region_cat = df.groupby(["Region","Product Category"])["Revenue"].sum().reset_index()
fig2 = px.treemap(region_cat, path=["Region","Product Category"],
                   values="Revenue", color="Revenue", color_continuous_scale=SEQ_COLORS,
                   title="Revenue Treemap: Region → Category")
L(fig2, "Revenue Treemap: Region → Category")
fig2.show()

# Stacked Bar — Annual Revenue by Category
yr_cat = df.groupby(["Year","Product Category"])["Revenue"].sum().reset_index()
fig3 = px.bar(yr_cat, x="Year", y="Revenue", color="Product Category",
               barmode="stack", color_discrete_sequence=CAT_COLORS)
L(fig3, "Annual Revenue Composition by Product Category")
fig3.show()

In [6]:
# ═══ PAGE 4: DATA RELATIONSHIP ═════════════════════════════════════════════════
num_cols_5 = ["Quantity Sold","Unit Price","Discount (%)","Total Sales","Profit Margin"]

# Correlation Heatmap
corr = df[num_cols_5].corr()
fig = go.Figure(go.Heatmap(z=corr.values, x=corr.columns.tolist(), y=corr.columns.tolist(),
                             colorscale=DIVERG, zmid=0,
                             text=np.round(corr.values,2), texttemplate="%{text}"))
L(fig, "Pearson Correlation Heatmap")
fig.show()

# Scatter — Discount vs Profit Margin with OLS trendline
fig2 = px.scatter(df.sample(min(2000,len(df)), random_state=42),
                   x="Discount (%)", y="Profit Margin",
                   color="Product Category", size="Total Sales",
                   color_discrete_sequence=CAT_COLORS, opacity=0.65,
                   trendline="ols", trendline_scope="overall",
                   trendline_color_override=PALETTE["accent3"])
L(fig2, "Discount (%) vs Profit Margin — OLS Trend")
fig2.show()

In [7]:
# ═══ PAGE 5: DATA COMPARISON ═══════════════════════════════════════════════════
# Regional Revenue by Year
reg_yr = df.groupby(["Region","Year"])["Revenue"].sum().reset_index()
fig = px.bar(reg_yr, x="Year", y="Revenue", color="Region", barmode="group",
              color_discrete_sequence=CAT_COLORS)
L(fig, "Regional Revenue Comparison by Year")
fig.show()

# YoY Revenue + Growth Rate (dual axis)
yearly = df.groupby("Year")["Revenue"].sum().reset_index()
yearly["YoY%"] = yearly["Revenue"].pct_change()*100
fig2 = make_subplots(specs=[[{"secondary_y":True}]])
fig2.add_trace(go.Bar(x=yearly["Year"], y=yearly["Revenue"],
                       name="Revenue", marker_color=PALETTE["accent1"], opacity=0.8), secondary_y=False)
fig2.add_trace(go.Scatter(x=yearly["Year"], y=yearly["YoY%"],
                           name="YoY Growth %", mode="lines+markers",
                           line=dict(color=PALETTE["accent3"], width=2.5)), secondary_y=True)
fig2.update_yaxes(title_text="Revenue ($)", secondary_y=False, gridcolor=PALETTE["grid"])
fig2.update_yaxes(title_text="YoY Growth %", secondary_y=True, gridcolor=PALETTE["grid"])
L(fig2, "Year-over-Year Revenue & Growth Rate")
fig2.show()

In [8]:
# ═══ MULTIVARIATE: 3D + Parallel Coords + Sunburst ══════════════════════════════
# 3D Scatter
s3d = df.dropna(subset=["Unit Price","Discount (%)","Revenue","Product Category"])
s3d = s3d.sample(min(1000,len(s3d)), random_state=9)
fig = px.scatter_3d(s3d, x="Unit Price", y="Discount (%)", z="Revenue",
                     color="Product Category", size="Quantity Sold",
                     color_discrete_sequence=CAT_COLORS, opacity=0.7)
L(fig, "3D Scatter: Unit Price × Discount × Revenue")
fig.show()

# Parallel Coordinates
num_cols_5 = ["Quantity Sold","Unit Price","Discount (%)","Total Sales","Profit Margin"]
fig2 = px.parallel_coordinates(
    df[num_cols_5+["Year"]].dropna().sample(min(1500,len(df)), random_state=5),
    color="Year", color_continuous_scale=SEQ_COLORS, dimensions=num_cols_5)
L(fig2, "Parallel Coordinates — All Numeric KPIs by Year")
fig2.show()

# Sunburst
fig3 = px.sunburst(df, path=["Region","Product Category","Payment Method"],
                    values="Revenue", color="Revenue", color_continuous_scale=SEQ_COLORS)
L(fig3, "Revenue Sunburst: Region → Category → Payment Method")
fig3.show()

In [9]:
# ═══ STATISTICAL ANALYSIS ══════════════════════════════════════════════════════
num_cols_5 = ["Quantity Sold","Unit Price","Discount (%)","Total Sales","Profit Margin"]

# ── ANOVA: Revenue ~ Region
groups = [df[df["Region"]==r]["Revenue"].dropna() for r in df["Region"].dropna().unique()]
f_stat, p_anova = stats.f_oneway(*groups)
print(f"ANOVA  F={f_stat:.2f}  p={p_anova:.4e}  →  {'Significant ✔' if p_anova<0.05 else 'Not significant ✘'}")

# ── Pearson correlations
pairs = [("Discount (%)","Profit Margin"),("Unit Price","Total Sales"),
         ("Quantity Sold","Total Sales"),("Discount (%)","Total Sales")]
for a,b in pairs:
    r, p = pearsonr(df[a].dropna(), df[b].dropna()[:len(df[a].dropna())])
    print(f"  {a:20s} ↔ {b:20s}  r={r:+.4f}  p={p:.3e}  {'✔' if p<0.05 else '✘'}")

# ── OLS Regression
ols_df = df[["Revenue","Unit Price","Quantity Sold","Discount (%)"]].dropna()
X = sm.add_constant(ols_df[["Unit Price","Quantity Sold","Discount (%)"]])
model = sm.OLS(ols_df["Revenue"], X).fit()
print(f"\nOLS  R²={model.rsquared:.4f}  Adj R²={model.rsquared_adj:.4f}")
print(model.summary().tables[1])

# ── OLS Coefficient Plot
coef = model.params.drop("const"); ci = model.conf_int().drop("const")
fig = go.Figure(go.Bar(
    x=coef.index, y=coef.values,
    error_y=dict(type="data", array=(ci[1]-coef).values,
                 arrayminus=(coef-ci[0]).values, visible=True, color=PALETTE["accent3"]),
    marker_color=[PALETTE["accent4"] if v>0 else PALETTE["accent3"] for v in coef.values]
))
L(fig, f"OLS Regression Coefficients  (R²={model.rsquared:.3f})")
fig.show()

# ── Q-Q Plot
sample_rev = df["Revenue"].dropna().sample(min(500,len(df)), random_state=42)
(osm, osr), (slope, intercept, _) = stats.probplot(sample_rev, dist="norm")
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=list(osm), y=list(osr), mode="markers",
                           marker=dict(color=PALETTE["accent4"], size=4, opacity=0.7), name="Data"))
fig2.add_trace(go.Scatter(x=[min(osm),max(osm)],
                           y=[slope*min(osm)+intercept,slope*max(osm)+intercept],
                           mode="lines", line=dict(color=PALETTE["accent3"],width=2), name="Normal"))
L(fig2, "Q-Q Plot: Revenue Normality Check")
fig2.show()

ANOVA  F=2.71  p=2.8487e-02  →  Significant ✔
  Discount (%)         ↔ Profit Margin         r=-0.1203  p=1.435e-17  ✔
  Unit Price           ↔ Total Sales           r=+0.6637  p=0.000e+00  ✔
  Quantity Sold        ↔ Total Sales           r=+0.6179  p=0.000e+00  ✔
  Discount (%)         ↔ Total Sales           r=-0.1203  p=1.435e-17  ✔

OLS  R²=0.8637  Adj R²=0.8636
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const         -1012.2799     17.060    -59.336      0.000   -1045.725    -978.834
Unit Price        4.7178      0.036    130.747      0.000       4.647       4.789
Quantity Sold   218.3352      1.784    122.398      0.000     214.838     221.832
Discount (%)    -14.0488      0.593    -23.686      0.000     -15.212     -12.886


In [10]:
# ═══ ADVANCED INSIGHTS ══════════════════════════════════════════════════════════
# Top 10 Products by Revenue
top_prods = (df.groupby("Product Name")["Revenue"].sum()
               .reset_index().sort_values("Revenue", ascending=True).tail(10))
fig = px.bar(top_prods, x="Revenue", y="Product Name", orientation="h",
              color="Revenue", color_continuous_scale=SEQ_COLORS,
              text=top_prods["Revenue"].map("${:,.0f}".format))
L(fig, "Top 10 Products by Revenue")
fig.update_traces(textposition="outside")
fig.show()

# Pareto Chart
prod_rev  = df.groupby("Product Name")["Revenue"].sum().sort_values(ascending=False).reset_index()
prod_rev["Cum%"] = prod_rev["Revenue"].cumsum() / prod_rev["Revenue"].sum() * 100
prod_rev["Rank"] = range(1, len(prod_rev)+1)
fig2 = make_subplots(specs=[[{"secondary_y":True}]])
fig2.add_trace(go.Bar(x=prod_rev["Rank"], y=prod_rev["Revenue"],
                       name="Revenue", marker_color=PALETTE["accent1"], opacity=0.75), secondary_y=False)
fig2.add_trace(go.Scatter(x=prod_rev["Rank"], y=prod_rev["Cum%"], name="Cumulative %",
                           mode="lines", line=dict(color=PALETTE["accent3"],width=2.5)), secondary_y=True)
fig2.update_yaxes(title_text="Revenue ($)", secondary_y=False, gridcolor=PALETTE["grid"])
fig2.update_yaxes(title_text="Cumulative %", secondary_y=True, gridcolor=PALETTE["grid"])
L(fig2, "Pareto Chart — Revenue Concentration by Product")
fig2.show()

# Quarterly Cohort
qtr_rev = df.groupby(["Year","Quarter"])["Revenue"].sum().reset_index()
fig3 = px.line(qtr_rev, x="Quarter", y="Revenue", color=qtr_rev["Year"].astype(str),
                markers=True, color_discrete_sequence=SEQ_COLORS)
L(fig3, "Quarterly Revenue by Year — Cohort Trend")
fig3.show()

print("\n✅ Full analysis complete! See amazon_sales_dashboard.html for the interactive dashboard.")


✅ Full analysis complete! See amazon_sales_dashboard.html for the interactive dashboard.
